<a href="https://colab.research.google.com/github/SanjaraT/Sentiment-Analysis-BERT/blob/main/Bengali_sentiment_analysis_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers

In [ ]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer
from transformers import BertForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Load Dataset

In [ ]:
def load_data(pos_path, neg_path):
  with open(pos_path, encoding='utf-8') as f:
    pos = f.readlines()

  with open(neg_path, encoding='utf-8') as f:
    neg = f.readlines()

    texts = pos + neg
    labels = [1]*len(pos) + [0]*len(neg)

    return pd.DataFrame({'text': texts, 'label': labels})

df = load_data(
    "/content/drive/MyDrive/Beng_sent_data/all_positive_8500.txt",
    "/content/drive/MyDrive/Beng_sent_data/all_positive_8500.txt"
)

df = df.sample(frac=1).reset_index(drop=True)
df.head()

,text,label
0,তুমি বেষ্ট এক্টর বস @আরিফিন_নিশু#Best Aktor\n,0
1,"অসাধারণ খুব ভালো লাগলো,,,,,,,,,,,,এমন সুন্দর ...",0
2,শেষে ২জন এর মাঝে মিল দিলে ভালো হইতো\n,0
3,এই natoke mir sbbir থাকলে আরো সুন্দর হয়তো\n,1
4,অসাধারননাটক দেখলাম নাকি মুভি দেখলাম।কিছুটা Ka...,0


In [ ]:
df.shape

(11807, 2)

# Train-Test

In [ ]:
from numpy import random
from sklearn.model_selection import train_test_split

train_text, test_text, train_label, test_label = train_test_split(
    df['text'].tolist(),
    df['label'].tolist(),
    test_size = 0.1,
    random_state = 42
)

# Tokenization

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

def tokenize(texts):
  return tokenizer(
      texts,
      padding=True,
      truncation=True,
      max_length=512,
      return_tensors='pt'
  )

train_encodings = tokenize(train_text)
test_encodings = tokenize(test_text)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Custom Dataset Class

In [ ]:
class SentimentDataset(Dataset):
  def __init__(self, encodings, labels):
    self.encodings = encodings
    self.labels = labels

  def __getitem__(self, idx):
    item = {key: val[idx] for key, val in self.encodings.items()}
    item['labels'] = torch.tensor(self.labels[idx])
    return item

  def __len__(self):
    return len(self.labels)

In [ ]:
# Dataset Object
train_dataset = SentimentDataset(train_encodings, train_label)
test_dataset = SentimentDataset(test_encodings, test_label)

In [ ]:
# Dataloader object
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

# Model

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-multilingual-cased',
     num_labels=2
    )

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


#

In [ ]:
# Moving Model to GPU
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
print(device)

cuda


In [ ]:
optimizer = AdamW(model.parameters(), lr=2e-5)

# Training Pipeline

In [ ]:
from tqdm import tqdm

for epoch in range(3):
  model.train()
  loop = tqdm(train_loader, leave=True)
  total_loss = 0

  for batch in loop:
    optimizer.zero_grad()


    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    outputs = model(
        input_ids,
        attention_mask=attention_mask,
        labels=labels
    )


    loss = outputs.loss
    total_loss += loss.item()

    loss.backward()
    optimizer.step()

    loop.set_description(f"Epoch {epoch+1}")
    loop.set_postfix(loss=loss.item())

print(f"Epoch {epoch+1} finished | Avg Loss: {total_loss/len(train_loader):.4f}")

Epoch 3: 100%|██████████| 957/957 [24:40<00:00,  1.55s/it, loss=0.7]

Epoch 3 finished | Avg Loss: 0.6949


# Evaluation

In [ ]:
model.eval()

preds = []
true_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        predictions = torch.argmax(logits, dim=1).cpu().numpy()

        preds.extend(predictions)
        true_labels.extend(batch['labels'].numpy())

print("Accuracy:", accuracy_score(true_labels, preds))
print(classification_report(true_labels, preds))

Accuracy: 0.5170588235294118
              precision    recall  f1-score   support

           0       0.52      1.00      0.68       879
           1       0.00      0.00      0.00       821

    accuracy                           0.52      1700
   macro avg       0.26      0.50      0.34      1700
weighted avg       0.27      0.52      0.35      1700



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Inference

In [ ]:
def predict(text):
    model.eval()

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

    outputs = model(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()

    return "Positive" if pred == 1 else "Negative"

# Test
print(predict("নাটকটি অসাধারণ ছিল"))
print(predict("পুরা সময়টাই নষ্ট"))

Negative
Negative
